# Lab 1: Data Architecture & Governance

## Goal

Experience how Unity Catalog provides governance, discoverability, and lineage for the financial
intelligence platform.

## What you'll do

1. Explore the shared catalog structure (schemas, volumes, tables)
2. Tag your personal schema and observe how tags appear in the Catalog Explorer
3. Trace data lineage from a gold table back to the source PDF
4. Review the access controls on the shared volume

---

In [0]:
%run ../utils/config

## Part 1: Explore the catalog structure

**First, explore in the UI:**
1. Click **Catalog** in the left sidebar
2. Expand `vectorcatalog01` → `00_shared`
3. Notice: schemas, volumes, and tables are all first-class objects with metadata

Now run the same exploration via SQL:

In [0]:
# What schemas exist in the workshop catalog?
display(spark.sql(f"SHOW SCHEMAS IN {catalog}"))

In [0]:
# What volumes hold our raw documents?
display(spark.sql(f"SHOW VOLUMES IN {catalog}.{shared_schema}"))

In [0]:
# What tables did the pipeline produce?
display(spark.sql(f"SHOW TABLES IN {catalog}.{shared_schema}"))

In [0]:
# Inspect the gold table schema — notice the business-friendly column names and comments
display(spark.sql(f"DESCRIBE TABLE EXTENDED {catalog}.{shared_schema}.financial_docs_gold"))

### Setting a default catalog and schema
In the preceding cells, we explicitly used the three-level namespace for catalog.schema.table.  the `USE CATALOG` and `USE SCHEMA` SQL commands set a default catalog and schema for all following commands in the notebook.

In [0]:
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")
print(f"✓ Default catalog.schema set to ", spark.catalog.currentCatalog(), spark.catalog.currentDatabase())

## Part 2: Tag your personal schema

Tags make data assets discoverable across the organization.
Add a tag to your personal schema so it shows up in catalog searches.

### Adding tables and tags

In [0]:
# Add tags to your personal schema
# The schema name is required in the ALTER SCHEMA statement, but the catalog is optional
spark.sql(f"""
    ALTER SCHEMA {schema}
    SET TAGS ('workshop_name' = 'robust-ai-data-platforms', 'participant' = '{username}')
""")
print(f"✓ Tags added to {catalog}.{schema}")

In [0]:
# Create a sample table in your schema to explore tagging at the table level
spark.sql(f"""
    CREATE OR REPLACE TABLE my_analysis AS
        SELECT company, fiscal_period, ai_investment_category, COUNT(*) AS doc_count
        FROM {shared_schema}.company_ai_investment_summary
        GROUP BY ALL
        ORDER BY company, fiscal_period
""")
print(f"✓ Created {catalog}.{schema}.my_analysis")

In [0]:
# Tag the table with a business domain and classification
spark.sql(f"""
    ALTER TABLE my_analysis
    SET TAGS (
        'domain'         = 'finance',
        'classification' = 'public',
        'contains_ai'    = 'true'
    )
""")
print("✓ Table tags set")

In [0]:
# Verify tags are visible — scroll to the end of DESCRIBE EXTENDED output
display(spark.sql(f"DESCRIBE EXTENDED my_analysis"))

### Browse your new table
**In the UI:**
1. Open **Catalog Explorer** in the left sidebar
1. Find the `my_analysis` table in your personal schema
1. Note the **Tags** you just applied in the details on the right.
1. Note how the column comments propagated from the source of the CTAS statement.
1. Use **AI** to generate and accept a table description and column comments.
<img src=../utils/_resources/my_analysisTable.png>

## Part 3: Explore data lineage

Unity Catalog automatically tracks how data flows between tables.

**In the UI:**
1. Navigate to **Catalog** → `vectorcatalog01` → `00_shared` → `financial_docs_silver`
2. Click the **Lineage** tab - you should see tables, views, pipelines, jobs, and notebooks associated with this table.
3. Click the **See lineage graph** button.
4. Explore and expand the lineage graph.

- How far back can you trace lineage?
- Is lineage at the table level, or column level?

This lineage was captured automatically — no manual documentation required.
<img src=../utils/_resources/Lineage.png>

## Part 4: Access control

Let's inspect the grants on the shared Volume and tables.

In [0]:
# Who can read the shared financial documents?
display(spark.sql(f"SHOW GRANTS ON VOLUME {shared_schema}.financial_docs_raw"))

In [0]:
# Who can read the gold table?
display(spark.sql(f"SHOW GRANTS ON TABLE {shared_schema}.financial_docs_gold"))
# Note that the SELECT GRANT is inherited from the parent schema

In [0]:
# Only you own your personal schema — others cannot read it by default
display(spark.sql(f"SHOW GRANTS ON SCHEMA {schema}"))

## Key Takeaways

1. **Unity Catalog is the governance layer** — schemas, volumes, tables, and even ML models all live here with consistent access control and tagging
2. **Lineage is automatic** — the platform tracks data provenance from raw PDF to insight without any manual effort
4. **Everything is code** — schema creation, grants, and tags can all be scripted, versioned, and reviewed

---


**Next:** Lab 2 — Scalable pipelines for unstructured data

→ Open [03_unstructured_pipelines]($./03_unstructured_pipelines)